# 14 · RAG (Retrieval-Augmented Generation) en Python: de cero a un nivel sólido

Este cuaderno enseña **RAG** desde los fundamentos. Primero construiremos un sistema RAG **a mano**, con las piezas mínimas, para entender cada componente; después veremos cómo **LangChain** y **LlamaIndex** encapsulan exactamente lo mismo (enlaza con los cuadernos `11_langchain` y `13_llamaindex`).

**Stack local y sin servicios de pago:**
- Embeddings: `sentence-transformers` (modelo `all-MiniLM-L6-v2`).
- Almacén vectorial: NumPy puro, luego `faiss-cpu` y `chromadb`.
- Generación: **Ollama** en local (`ollama`, modelo `llama3.2`).

## Índice

0. Requisitos e instalación
1. Introducción: qué es RAG y qué problema resuelve
2. Fundamento 1 — Embeddings
3. Fundamento 2 — Chunking (troceado)
4. Fundamento 3 — Almacén vectorial casero con NumPy
5. Fundamento 4 — Generación aumentada
6. Pipeline RAG completo de principio a fin
7. Almacén vectorial real con FAISS
8. Almacén vectorial con ChromaDB
9. Mejorar la recuperación
10. Evaluación de un sistema RAG
11. Cómo lo hacen los frameworks (LangChain y LlamaIndex)
12. Buenas prácticas y errores comunes
13. Chuleta de sintaxis
14. Drills
- Soluciones propuestas


## 0. Requisitos e instalación

Ejecute la siguiente celda una sola vez para instalar las dependencias. La **generación** de respuestas requiere [Ollama](https://ollama.com) instalado y en marcha, con el modelo descargado (`ollama pull llama3.2`).

Las secciones de **recuperación** (embeddings, chunking, búsqueda) funcionan sin Ollama. Solo la generación (secciones 5, 6 y 11) necesita el servicio activo. El resto del stack es local y no requiere claves ni servicios de pago.


In [ ]:
# Instalación de dependencias (ejecutar una sola vez).
# La generación de respuestas requiere Ollama en local: https://ollama.com
# Tras instalar Ollama, descargue el modelo con:  ollama pull llama3.2
%pip install sentence-transformers faiss-cpu chromadb ollama numpy

# Comprobación rápida del entorno.
import numpy as np
print("NumPy:", np.__version__)
try:
    import sentence_transformers
    print("sentence-transformers:", sentence_transformers.__version__)
except Exception as e:
    print("sentence-transformers no disponible todavía:", e)


## 1. Introducción: qué es RAG y qué problema resuelve

Un modelo de lenguaje (LLM) solo "sabe" lo que vio durante su entrenamiento. Esto plantea tres problemas:

- **Conocimiento congelado y genérico:** no conoce sus documentos privados ni los cambios recientes.
- **Alucinaciones:** cuando no sabe algo, tiende a inventar una respuesta plausible pero falsa.
- **Falta de trazabilidad:** no puede citar de dónde sale la información.

**RAG (Retrieval-Augmented Generation)** resuelve esto sin reentrenar el modelo: antes de responder, **recupera** los fragmentos relevantes de un corpus propio y los **inyecta** en el prompt. El modelo genera la respuesta basándose en ese contexto.

### Las dos fases del pipeline

```
FASE OFFLINE (indexación, una sola vez)
  documentos --> chunking --> embeddings --> [ALMACEN VECTORIAL]

FASE ONLINE (en cada consulta)
  pregunta --> embedding --> busqueda top-k --> contexto
                                                  |
                             pregunta + contexto --> LLM --> respuesta citada
```

La clave: **la calidad de la respuesta depende sobre todo de la calidad de la recuperación**. Si no se recupera el fragmento correcto, el LLM no podrá responder bien.


In [ ]:
# Representación conceptual de las dos fases del pipeline RAG.
pipeline_offline = [
    "1. Cargar documentos (corpus)",
    "2. Trocear en fragmentos (chunking)",
    "3. Calcular embeddings de cada fragmento",
    "4. Guardar vectores + texto en el almacén vectorial (indexación)",
]
pipeline_online = [
    "1. Recibir la pregunta del usuario",
    "2. Calcular el embedding de la pregunta",
    "3. Buscar los fragmentos más similares (recuperación / retrieval)",
    "4. Construir un prompt inyectando esos fragmentos como contexto",
    "5. Generar la respuesta con el LLM (generación)",
]
print("FASE OFFLINE (indexación):")
for paso in pipeline_offline:
    print("  ", paso)
print("\nFASE ONLINE (consulta):")
for paso in pipeline_online:
    print("  ", paso)


### Ejercicio 1

Con sus propias palabras, enumere tres problemas que RAG ayuda a mitigar y proponga un caso de uso concreto en un entorno con documentación confidencial (por ejemplo, normativa interna o expedientes). No hace falta código.


## 2. Fundamento 1 — Embeddings

Un **embedding** es un vector denso (una lista de números en coma flotante) que representa el **significado** de un texto. La propiedad fundamental: **textos con significado parecido producen vectores cercanos** en el espacio vectorial.

Para medir cercanía se usa la **similitud coseno**, que mide el ángulo entre dos vectores:
- valor cercano a **1** → muy parecidos,
- cercano a **0** → sin relación,
- cercano a **-1** → opuestos.

El modelo `all-MiniLM-L6-v2` convierte cada texto en un vector de **384 dimensiones**. Es rápido, ligero y suficiente para aprender.


In [ ]:
from sentence_transformers import SentenceTransformer, util

# Modelo compacto y rápido. Produce vectores de 384 dimensiones.
# La primera ejecución descarga el modelo (unos 90 MB).
model = SentenceTransformer("all-MiniLM-L6-v2")

frases = [
    "El gato duerme en el sofá.",
    "Un felino descansa sobre el diván.",
    "La factura vence el día 30.",
]
vecs = model.encode(frases)
print("Forma de la matriz de embeddings:", vecs.shape)   # (3, 384)
print("Primeras 8 dimensiones del primer vector:\n", vecs[0][:8])


In [ ]:
import numpy as np

def coseno(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print("gato vs felino (parecidas):", round(coseno(vecs[0], vecs[1]), 3))
print("gato vs factura (distintas):", round(coseno(vecs[0], vecs[2]), 3))

# Equivalente con la utilidad de sentence-transformers (devuelve una matriz de similitudes).
print("\nMatriz cos_sim (util):\n", util.cos_sim(vecs, vecs))


### Ejercicio 2

Cree una lista de cinco frases sobre temas variados (trabajo, ocio, animales...). Calcule sus embeddings y, dada la consulta `"reunión de trabajo"`, ordene las frases de mayor a menor similitud coseno con la consulta.


## 3. Fundamento 2 — Chunking (troceado)

Los documentos suelen ser largos. Se **trocean** en fragmentos (*chunks*) por varias razones:

- Los modelos de embedding tienen un **límite de longitud** de entrada.
- Fragmentos pequeños producen embeddings más **precisos** y recuperaciones más **focalizadas**.
- Un chunk debe ser lo bastante grande para contener una idea completa, pero no tanto que mezcle temas.

**Estrategias habituales:**
- **Tamaño fijo** (por caracteres o tokens): simple, pero puede cortar frases a la mitad.
- **Con solape (overlap):** los chunks comparten un tramo en la frontera para no perder contexto.
- **Por frases o párrafos:** respeta las unidades semánticas naturales.

No existe un tamaño óptimo universal: depende del tipo de documento y de las consultas.


In [ ]:
def trocear_fijo(texto, tam=200):
    """Trocea por número de caracteres, sin solape (la estrategia más simple)."""
    return [texto[i:i + tam] for i in range(0, len(texto), tam)]

ejemplo = ("La política de teletrabajo de Helios permite hasta tres días "
           "remotos por semana. La solicitud se tramita en el portal interno "
           "y requiere aprobación del responsable de equipo.")

for i, c in enumerate(trocear_fijo(ejemplo, 80)):
    print(f"[chunk {i}] {c!r}")


In [ ]:
import re

def trocear_con_solape(texto, tam=200, solape=40):
    """Trocea por caracteres con solape para no cortar ideas en la frontera."""
    chunks = []
    paso = tam - solape
    for i in range(0, len(texto), paso):
        chunks.append(texto[i:i + tam])
        if i + tam >= len(texto):
            break
    return chunks

def trocear_por_frases(texto, max_frases=2):
    """Agrupa frases completas; respeta mejor la unidad semántica."""
    frases = re.split(r"(?<=[.!?])\s+", texto.strip())
    return [" ".join(frases[i:i + max_frases]) for i in range(0, len(frases), max_frases)]

print("CON SOLAPE (tam=80, solape=20):")
for c in trocear_con_solape(ejemplo, 80, 20):
    print("  ", repr(c))

print("\nPOR FRASES (1 frase por chunk):")
for c in trocear_por_frases(ejemplo, 1):
    print("  ", repr(c))


### Ejercicio 3

Implemente una función `trocear_por_parrafos(texto)` que divida un texto por dobles saltos de línea (párrafos) y elimine los fragmentos vacíos. Pruébela con un texto de varios párrafos.


## 4. Fundamento 3 — Almacén vectorial casero con NumPy

Un **almacén vectorial** (vector store) guarda los embeddings de todos los chunks y permite buscar los más parecidos a una consulta. Antes de usar librerías especializadas, lo construiremos **a mano con NumPy** para no ocultar el mecanismo.

La idea:
1. Apilar todos los embeddings en una **matriz** de forma `(n_chunks, 384)`.
2. Para buscar, calcular la similitud coseno entre el embedding de la pregunta y **todas** las filas.
3. Devolver los **top-k** con mayor similitud.

Truco clave: si **normalizamos** los vectores (norma 1), la similitud coseno se reduce a un simple **producto escalar** (`matriz @ q`), que NumPy calcula de forma vectorizada y muy rápida.


In [ ]:
# Corpus de ejemplo: normativa interna ficticia de "Corporación Helios".
# Todo el conocimiento vive aquí, en memoria; no se descarga nada externo.
documentos = [
    "Corporación Helios es una empresa ficticia dedicada a la energía solar, "
    "fundada en 2018 con sede en Bilbao. Su plantilla ronda los 400 empleados.",

    "La política de teletrabajo de Helios permite hasta tres días remotos por "
    "semana. La solicitud se tramita en el portal interno y requiere aprobación "
    "del responsable de equipo con al menos 48 horas de antelación.",

    "El horario laboral estándar es de 8:00 a 16:00 de lunes a viernes. Existe "
    "flexibilidad de entrada entre las 7:30 y las 9:30, compensando la diferencia "
    "en la hora de salida.",

    "Las vacaciones anuales son de 23 días laborables. Se solicitan con un mínimo "
    "de 15 días de antelación y no se pueden acumular más de 5 días para el año "
    "siguiente.",

    "El proceso de alta de un proveedor nuevo exige validación fiscal, firma del "
    "acuerdo de confidencialidad y aprobación del departamento de compras. El plazo "
    "medio de alta es de siete días hábiles.",

    "La política de seguridad obliga a usar autenticación de doble factor en todos "
    "los sistemas corporativos. Las contraseñas caducan cada 90 días y deben tener "
    "al menos 12 caracteres.",

    "El comedor de la sede ofrece menú subvencionado. El precio para el empleado es "
    "de 4 euros y el horario de servicio es de 13:00 a 15:00.",
]
print(f"{len(documentos)} documentos cargados en el corpus.")


In [ ]:
# Indexación: troceamos cada documento y calculamos los embeddings de cada chunk.
def construir_indice(docs, tam=240, solape=40):
    chunks, meta = [], []
    for doc_id, doc in enumerate(docs):
        for pos, ch in enumerate(trocear_con_solape(doc, tam, solape)):
            chunks.append(ch)
            meta.append({"doc_id": doc_id, "pos": pos})
    matriz = model.encode(chunks, normalize_embeddings=True)  # vectores unitarios
    return chunks, meta, np.asarray(matriz, dtype=np.float32)

chunks, meta, matriz = construir_indice(documentos)
print("Nº de chunks:", len(chunks), "| forma de la matriz:", matriz.shape)

def buscar(pregunta, k=3):
    """Devuelve los k chunks más similares por coseno (vectores ya normalizados)."""
    q = model.encode([pregunta], normalize_embeddings=True)[0].astype(np.float32)
    # Con vectores unitarios, el producto escalar ES la similitud coseno.
    puntuaciones = matriz @ q
    top = np.argsort(-puntuaciones)[:k]
    return [(int(i), float(puntuaciones[i]), chunks[i]) for i in top]


In [ ]:
for idx, score, texto in buscar("¿Cuántos días puedo teletrabajar?", k=3):
    print(f"[{score:.3f}] chunk {idx}: {texto}")


### Ejercicio 4

Modifique la función `buscar` para que, además del índice, el score y el texto, devuelva el `doc_id` de cada chunk recuperado (disponible en la lista `meta`).


## 5. Fundamento 4 — Generación aumentada

Ya sabemos recuperar los fragmentos relevantes. Ahora los **inyectamos en el prompt** para que el LLM genere una respuesta anclada en ellos. Dos elementos son críticos:

- **Prompt de sistema:** instruye al modelo para que responda **solo** con el contexto y admita cuándo no tiene información. Esto reduce drásticamente las alucinaciones.
- **Manejo del caso "sin información":** si la recuperación no encuentra nada relevante, el sistema debe decirlo en lugar de inventar.

Usaremos **Ollama** en local. Con `temperature=0` las respuestas son deterministas y más fieles al contexto.


In [ ]:
SISTEMA = (
    "Eres un asistente que responde EXCLUSIVAMENTE con la información del "
    "contexto proporcionado. Si el contexto no contiene la respuesta, indica "
    "claramente que no dispones de información suficiente. Cita el número de "
    "fragmento entre corchetes."
)

def construir_prompt(pregunta, recuperados):
    """Inyecta los chunks recuperados en el mensaje de usuario."""
    bloques = [f"[{i}] {texto}" for i, _, texto in recuperados]
    contexto = "\n".join(bloques)
    usuario = (
        f"Contexto:\n{contexto}\n\n"
        f"Pregunta: {pregunta}\n"
        "Responde basándote solo en el contexto anterior."
    )
    return SISTEMA, usuario

sis, usr = construir_prompt("¿Cuántos días de vacaciones hay?", buscar("vacaciones", 3))
print(usr)


In [ ]:
import ollama

def generar(sistema, usuario, modelo="llama3.2"):
    """Llama a Ollama en local. Requiere el servicio arrancado y el modelo descargado."""
    respuesta = ollama.chat(
        model=modelo,
        messages=[
            {"role": "system", "content": sistema},
            {"role": "user", "content": usuario},
        ],
        options={"temperature": 0.0},  # respuestas deterministas y fieles al contexto
    )
    return respuesta["message"]["content"]

# Ejemplo (descomentar con Ollama en marcha):
# print(generar(sis, usr))


### Ejercicio 5

Cree una variante de `construir_prompt` que, cuando la lista de fragmentos recuperados esté vacía, genere un prompt que pida al modelo indicar claramente que no dispone de datos para responder.


## 6. Pipeline RAG completo de principio a fin

Unimos todas las piezas en una única función `rag(pregunta)` que:
1. **recupera** los chunks relevantes (el chunking y los embeddings ya se hicieron en la indexación),
2. comprueba un **umbral de similitud** para el caso "sin información",
3. **construye el prompt** con el contexto,
4. **genera** la respuesta con Ollama.

Esto es, en esencia, un sistema RAG completo funcionando sin ningún framework.


In [ ]:
def rag(pregunta, k=3, umbral=0.25, modelo="llama3.2"):
    """Pipeline RAG completo: recuperar -> aumentar -> generar.

    Devuelve (respuesta, recuperados). Si nada supera el umbral de similitud,
    responde que no hay información suficiente sin llamar al modelo.
    """
    recuperados = buscar(pregunta, k=k)
    if not recuperados or recuperados[0][1] < umbral:
        return "No dispongo de información suficiente para responder.", recuperados
    sistema, usuario = construir_prompt(pregunta, recuperados)
    return generar(sistema, usuario, modelo=modelo), recuperados


In [ ]:
# Ejecución de ejemplo (requiere Ollama en marcha; descomente para probar).
# respuesta, fuentes = rag("¿Cada cuánto caducan las contraseñas?")
# print("RESPUESTA:\n", respuesta)
# print("\nFUENTES USADAS:")
# for i, score, texto in fuentes:
#     print(f"  [{score:.3f}] {texto[:70]}...")

# Sin Ollama, podemos al menos ver qué se recuperaría:
print("Recuperación para la pregunta de ejemplo:")
for i, score, texto in buscar("¿Cada cuánto caducan las contraseñas?", k=3):
    print(f"  [{score:.3f}] {texto}")


### Ejercicio 6

Amplíe la función `rag` (o cree `rag_con_fuentes`) para que devuelva, junto a la respuesta, la lista de `doc_id` de los documentos que se han citado como fuente.


## 7. Almacén vectorial real con FAISS

**FAISS** (Facebook AI Similarity Search) es una librería especializada en búsqueda de vecinos más cercanos, muy eficiente cuando hay miles o millones de vectores. Con vectores **normalizados**, un índice `IndexFlatIP` (producto interno) es equivalente a nuestra búsqueda por coseno con NumPy.

Flujo: crear el índice con la dimensión, `add` de la matriz, y `search(consulta, k)` que devuelve distancias e índices.


In [ ]:
import faiss

dim = matriz.shape[1]
index = faiss.IndexFlatIP(dim)   # producto interno = coseno con vectores normalizados
index.add(matriz)                # añadimos toda la matriz de chunks
print("Vectores en el índice FAISS:", index.ntotal)

def buscar_faiss(pregunta, k=3):
    q = model.encode([pregunta], normalize_embeddings=True).astype(np.float32)
    D, I = index.search(q, k)     # D = scores (producto interno), I = índices
    return [(int(i), float(d), chunks[i]) for d, i in zip(D[0], I[0])]

for idx, score, texto in buscar_faiss("horario del comedor", 3):
    print(f"[{score:.3f}] {texto}")


### Ejercicio 7

Compruebe que FAISS y la versión NumPy son equivalentes: para una misma consulta, verifique que `buscar` y `buscar_faiss` devuelven los mismos índices de chunk.


## 8. Almacén vectorial con ChromaDB

**ChromaDB** es una base de datos vectorial con **persistencia** y **metadatos** integrados. A diferencia de FAISS, guarda el texto y los metadatos junto a los vectores, y permite **filtrar** por metadatos en la búsqueda (`where`).

Operaciones básicas: `get_or_create_collection`, `add`/`upsert` (ids, documents, embeddings, metadatas) y `query` (por texto o por embedding, con `n_results` y filtros).


In [ ]:
import chromadb

cliente = chromadb.PersistentClient(path="./chroma_helios")  # persistencia en disco
coleccion = cliente.get_or_create_collection("normativa_helios")

# add() acepta documentos en texto y calcularía embeddings por defecto, pero aquí
# pasamos NUESTROS embeddings (MiniLM) para mantener coherencia con el resto del cuaderno.
coleccion.upsert(
    ids=[f"chunk-{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=[v.tolist() for v in matriz],
    metadatas=[{"doc_id": m["doc_id"]} for m in meta],
)

# Como guardamos embeddings propios, consultamos con query_embeddings (no query_texts),
# para que el espacio vectorial de la consulta coincida con el de los documentos.
q = model.encode(["¿cuántos días de vacaciones?"], normalize_embeddings=True).tolist()
res = coleccion.query(query_embeddings=q, n_results=3)
for doc, dist in zip(res["documents"][0], res["distances"][0]):
    print(f"[{dist:.3f}] {doc}")


In [ ]:
# Filtrado por metadatos con `where`: restringimos la búsqueda a un documento concreto.
q = model.encode(["seguridad y contraseñas"], normalize_embeddings=True).tolist()
res = coleccion.query(query_embeddings=q, n_results=2, where={"doc_id": 5})
print("Resultados filtrados al doc_id=5:")
for doc in res["documents"][0]:
    print("  ", doc)


### Ejercicio 8

Añada un metadato `tema` a cada chunk (por ejemplo, `"rrhh"`, `"seguridad"`, `"compras"`) y realice una `query` filtrando por uno de esos temas con `where`.


## 9. Mejorar la recuperación

La recuperación es el eslabón que más afecta a la calidad final. Palancas habituales:

- **Número de chunks `k`:** pocos pueden dejar fuera la respuesta; demasiados introducen ruido y encarecen el prompt.
- **Re-ranking:** recuperar muchos candidatos y reordenarlos con un modelo más preciso (**cross-encoder**) que puntúa cada par (pregunta, chunk).
- **Búsqueda híbrida:** combinar recuperación **léxica** (BM25, coincidencia de palabras) con la **semántica** (vectorial). Útil cuando importan términos exactos (códigos, nombres propios).
- **Filtros por metadatos:** restringir la búsqueda a un subconjunto (departamento, fecha, tipo de documento).


In [ ]:
# 1) Efecto de k: más contexto puede ayudar o introducir ruido.
for k in (1, 3, 5):
    top = buscar("teletrabajo", k=k)
    print(f"k={k} -> mejor score {top[0][1]:.3f}, {len(top)} chunks recuperados")

# 2) Re-ranking (esbozo): reordenar los candidatos por un criterio secundario.
#    En producción se usa un cross-encoder (p.ej. 'cross-encoder/ms-marco-MiniLM-L-6-v2')
#    que puntúa cada par (pregunta, chunk) con mayor precisión que el coseno.
candidatos = buscar("teletrabajo", k=5)
reordenados = sorted(candidatos, key=lambda x: len(x[2]))  # ejemplo trivial: por longitud
print("Reordenados (ejemplo trivial):", [c[0] for c in reordenados])

# 3) Búsqueda híbrida (mención): combinar BM25 (léxico) con vectorial (semántico).
#    BM25 puntúa por coincidencia de palabras; su score se fusiona con el vectorial.
#    Librerías habituales: rank_bm25, o los retrievers híbridos de LangChain/LlamaIndex.


### Ejercicio 9

Para la consulta `"seguridad"`, pruebe valores de `k` de 1 a 6 e imprima el **score medio** de los chunks recuperados en cada caso. ¿Cómo evoluciona al aumentar `k`?


## 10. Evaluación de un sistema RAG

Evaluar RAG implica medir dos cosas por separado:

- **Calidad de la recuperación:** ¿los chunks recuperados contienen realmente la respuesta? (*context precision/recall*).
- **Calidad de la generación:**
  - **Fidelidad (faithfulness):** ¿la respuesta se apoya en el contexto o inventa?
  - **Relevancia:** ¿responde de verdad a la pregunta?
  - **Corrección:** ¿coincide con la respuesta esperada (*ground truth*)?

Existen herramientas como **`ragas`** que automatizan estas métricas usando un LLM como juez. Para empezar, una **evaluación manual** con un pequeño conjunto de preguntas y respuestas esperadas ya aporta muchísima señal.


In [ ]:
# Evaluación cualitativa manual de la RECUPERACIÓN.
casos = [
    {"pregunta": "¿Cuántos días de teletrabajo?", "esperado": "tres"},
    {"pregunta": "¿Precio del menú del comedor?", "esperado": "4 euros"},
]

def precision_contexto(pregunta, esperado, k=3):
    """Proporción de chunks recuperados que contienen la respuesta esperada."""
    recuperados = buscar(pregunta, k=k)
    aciertos = sum(1 for _, _, t in recuperados if esperado.lower() in t.lower())
    return aciertos / len(recuperados)

for c in casos:
    p = precision_contexto(c["pregunta"], c["esperado"])
    print(f"{c['pregunta']:38s} -> precisión de contexto: {p:.2f}")

# Nota: herramientas como `ragas` automatizan métricas de faithfulness,
# answer_relevancy y context_precision usando un LLM como juez.


### Ejercicio 10

Añada un tercer caso de prueba a la lista `casos` y calcule la **precisión de contexto media** sobre todo el conjunto usando `precision_contexto`.


## 11. Cómo lo hacen los frameworks

Todo lo que hemos construido a mano (chunking, embeddings, almacén, recuperación, prompt, generación) es lo que **LangChain** y **LlamaIndex** encapsulan en pocas líneas. Verlo tras haberlo hecho artesanalmente permite entender qué ocurre por debajo.

- **LangChain:** `VectorStore` + `retriever` + una cadena que combina contexto y prompt. Véase el cuaderno `11_langchain`.
- **LlamaIndex:** `VectorStoreIndex` + `query_engine`, orientado precisamente a RAG. Véase el cuaderno `13_llamaindex`.

Los ejemplos siguientes son esquemáticos: requieren instalar las integraciones correspondientes y tener Ollama en marcha.


In [ ]:
# Mismo RAG con LangChain (esquemático; requiere langchain-community, langchain-huggingface, langchain-ollama).
from langchain_community.vectorstores import FAISS as LC_FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

emb = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vs = LC_FAISS.from_texts(documentos, emb)             # indexación en una línea
retriever = vs.as_retriever(search_kwargs={"k": 3})

prompt = ChatPromptTemplate.from_template(
    "Responde solo con el contexto.\nContexto: {context}\nPregunta: {question}"
)
llm = ChatOllama(model="llama3.2", temperature=0)

def rag_langchain(pregunta):
    docs = retriever.invoke(pregunta)
    contexto = "\n".join(d.page_content for d in docs)
    return llm.invoke(prompt.format(context=contexto, question=pregunta)).content

# print(rag_langchain("¿Cuántos días de vacaciones hay?"))


In [ ]:
# Mismo RAG con LlamaIndex (esquemático; requiere llama-index y sus integraciones).
from llama_index.core import VectorStoreIndex, Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.ollama import Ollama

emb_li = HuggingFaceEmbedding(model_name="all-MiniLM-L6-v2")
llm_li = Ollama(model="llama3.2", request_timeout=120.0)

docs_li = [Document(text=t) for t in documentos]
indice_li = VectorStoreIndex.from_documents(docs_li, embed_model=emb_li)  # indexación
motor = indice_li.as_query_engine(llm=llm_li, similarity_top_k=3)

# print(motor.query("¿Cuántos días de vacaciones hay?"))


### Ejercicio 11

Modifique `rag_langchain` para que devuelva también los documentos fuente recuperados por el `retriever`, además del texto de la respuesta.


## 12. Buenas prácticas y errores comunes

- **Calidad del corpus:** basura entra, basura sale. Limpie, deduplique y estructure los documentos antes de indexar.
- **Tamaño de chunk:** ni demasiado grande (mezcla temas, diluye la señal) ni demasiado pequeño (fragmenta ideas). Ajústelo y mida.
- **Prompt de sistema estricto:** obligue a responder **solo** con el contexto y a admitir "no consta". Es la principal defensa contra las alucinaciones.
- **Privacidad de datos sensibles:** con documentación confidencial, valore ejecutar el modelo **en local** (como aquí con Ollama) para que los datos no salgan de la organización. Controle qué se indexa y quién consulta.
- **Coste y latencia:** más `k` y prompts más largos implican más tokens, más coste y más lentitud. Busque el mínimo `k` que mantenga la calidad.
- **Trazabilidad:** cite siempre las fuentes; facilita la auditoría y la confianza del usuario.
- **Evalúe de forma continua:** mantenga un conjunto de preguntas de referencia y mida el efecto de cada cambio.


### Ejercicio 12

Redacte un prompt de sistema que **prohíba explícitamente inventar** y exija responder exactamente `"No consta en la documentación"` cuando el dato no aparezca en el contexto. Razone qué respondería el sistema ante una pregunta fuera del corpus.


## 13. Chuleta de sintaxis

### Embeddings y similitud
| Tarea | Código |
|---|---|
| Cargar modelo | `model = SentenceTransformer('all-MiniLM-L6-v2')` |
| Codificar textos | `vecs = model.encode(chunks)` |
| Codificar normalizado | `model.encode(x, normalize_embeddings=True)` |
| Dimensión del vector | `model.get_sentence_embedding_dimension()` |
| Similitud coseno (util) | `cos = util.cos_sim(q, vecs)` |
| Coseno con NumPy (normalizado) | `sim = matriz @ q` |

### Almacenes vectoriales
| Almacén | Código |
|---|---|
| NumPy top-k | `top = np.argsort(-(matriz @ q))[:k]` |
| FAISS crear | `index = faiss.IndexFlatIP(dim)` |
| FAISS añadir/buscar | `index.add(vecs); D, I = index.search(q, k)` |
| Chroma colección | `col = cliente.get_or_create_collection('x')` |
| Chroma añadir | `col.add(ids=..., documents=..., embeddings=..., metadatas=...)` |
| Chroma consultar (texto) | `col.query(query_texts=['...'], n_results=3)` |
| Chroma con filtro | `col.query(query_embeddings=q, n_results=3, where={'tema': 'rrhh'})` |

### Generación con Ollama
| Tarea | Código |
|---|---|
| Chat | `ollama.chat(model='llama3.2', messages=[...])` |
| Mensajes | `[{'role':'system','content':contexto}, {'role':'user','content':pregunta}]` |
| Determinista | `options={'temperature': 0.0}` |
| Texto de salida | `resp['message']['content']` |

### Frameworks (equivalente en pocas líneas)
| Framework | RAG |
|---|---|
| LangChain | `vs = FAISS.from_texts(docs, emb); r = vs.as_retriever(); r.invoke(pregunta)` |
| LlamaIndex | `idx = VectorStoreIndex.from_documents(docs); idx.as_query_engine().query(pregunta)` |


## 14. Drills

Ejercicios cortos de repetición para afianzar la mecánica. Resuelva cada uno en la celda vacía correspondiente. Las soluciones están al final del cuaderno.

1. Cargar `all-MiniLM-L6-v2` e imprimir la dimensión de sus vectores.
2. Codificar `"energía solar"` y mostrar la norma del vector antes y después de normalizar.
3. Escribir una función `coseno(a, b)` con NumPy.
4. Trocear un texto de 500 caracteres en chunks de 100 sin solape y contar cuántos salen.
5. Trocear el mismo texto con solape 25 y comparar el número de chunks.
6. Construir la matriz de embeddings de 4 frases e imprimir su forma.
7. Dada una consulta, devolver el índice del chunk más similar con `np.argmax`.
8. Construir un `IndexFlatIP` de FAISS, añadir la matriz e imprimir `ntotal`.
9. Crear una colección Chroma en memoria (`EphemeralClient`) y añadir 3 documentos.
10. Hacer una `query` a esa colección y mostrar el documento más cercano.
11. Escribir `construir_prompt_min(pregunta, contexto)` que devuelva un único string.
12. Escribir la llamada `ollama.chat` con mensajes de sistema y usuario (sin ejecutarla).


In [ ]:
# Drill 1: carga el modelo all-MiniLM-L6-v2 e imprime la dimensión de sus vectores.


In [ ]:
# Drill 2: codifica "energía solar" y muestra su norma antes y después de normalizar.


In [ ]:
# Drill 3: escribe una función coseno(a, b) con NumPy.


In [ ]:
# Drill 4: trocea un texto de 500 caracteres en chunks de 100 sin solape y cuéntalos.


In [ ]:
# Drill 5: trocea el mismo texto con solape 25 y compara el número de chunks.


In [ ]:
# Drill 6: construye la matriz de embeddings de 4 frases e imprime su forma.


In [ ]:
# Drill 7: devuelve el índice del chunk más similar a una consulta con np.argmax.


In [ ]:
# Drill 8: construye un IndexFlatIP de FAISS, añade la matriz e imprime ntotal.


In [ ]:
# Drill 9: crea una colección Chroma en memoria (EphemeralClient) y añade 3 documentos.


In [ ]:
# Drill 10: haz una query a esa colección y muestra el documento más cercano.


In [ ]:
# Drill 11: escribe construir_prompt_min(pregunta, contexto) que devuelva un único string.


In [ ]:
# Drill 12: escribe la llamada ollama.chat con mensajes de sistema y usuario (sin ejecutarla).


## Soluciones propuestas

A continuación se ofrecen soluciones comentadas a los ejercicios y a los drills. Consúltelas solo después de intentar resolverlos por su cuenta. Algunas celdas requieren Ollama o las integraciones de los frameworks para ejecutarse por completo.


In [ ]:
# --- Soluciones a los ejercicios 1 a 6 ---

# Ejercicio 1 (conceptual):
# Problemas que mitiga RAG:
#  - Conocimiento desactualizado: se actualiza cambiando el corpus, sin reentrenar.
#  - Alucinaciones: la respuesta se ancla en fragmentos reales y verificables.
#  - Falta de trazabilidad: se pueden citar las fuentes recuperadas.
# Caso de uso: asistente sobre normativa interna o documentación confidencial.

# Ejercicio 2:
frases_ej2 = [
    "La reunión de trabajo empieza a las nueve.",
    "El gato duerme al sol.",
    "Preparé el informe trimestral para el comité.",
    "Las vacaciones en la playa fueron relajantes.",
    "El equipo se reúne para planificar el proyecto.",
]
vecs_ej2 = model.encode(frases_ej2, normalize_embeddings=True)
q_ej2 = model.encode(["reunión de trabajo"], normalize_embeddings=True)[0]
orden = np.argsort(-(vecs_ej2 @ q_ej2))
for i in orden:
    print(f"{float(vecs_ej2[i] @ q_ej2):.3f}  {frases_ej2[i]}")

# Ejercicio 3:
def trocear_por_parrafos(texto):
    return [p.strip() for p in texto.split("\n\n") if p.strip()]

# Ejercicio 4:
def buscar_con_doc(pregunta, k=3):
    q = model.encode([pregunta], normalize_embeddings=True)[0].astype(np.float32)
    puntuaciones = matriz @ q
    top = np.argsort(-puntuaciones)[:k]
    return [(int(i), meta[i]["doc_id"], float(puntuaciones[i]), chunks[i]) for i in top]

# Ejercicio 5:
def construir_prompt_v2(pregunta, recuperados):
    if not recuperados:
        return SISTEMA, (f"No hay contexto disponible. Pregunta: {pregunta}. "
                         "Indica que no dispones de datos para responder.")
    return construir_prompt(pregunta, recuperados)

# Ejercicio 6:
def rag_con_fuentes(pregunta, k=3):
    respuesta, recuperados = rag(pregunta, k=k)
    doc_ids = sorted({meta[i]["doc_id"] for i, _, _ in recuperados})
    return respuesta, doc_ids


In [ ]:
# --- Soluciones a los ejercicios 7 a 12 ---

# Ejercicio 7:
consulta = "seguridad y contraseñas"
np_idx = [i for i, _, _ in buscar(consulta, 3)]
fa_idx = [i for i, _, _ in buscar_faiss(consulta, 3)]
print("NumPy:", np_idx, "| FAISS:", fa_idx, "| coinciden:", np_idx == fa_idx)

# Ejercicio 8:
temas = ["general", "rrhh", "rrhh", "rrhh", "compras", "seguridad", "servicios"]
coleccion.upsert(
    ids=[f"chunk-{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=[v.tolist() for v in matriz],
    metadatas=[{"doc_id": meta[i]["doc_id"], "tema": temas[meta[i]["doc_id"]]}
               for i in range(len(chunks))],
)
q8 = model.encode(["contraseñas"], normalize_embeddings=True).tolist()
print(coleccion.query(query_embeddings=q8, n_results=2,
                      where={"tema": "seguridad"})["documents"])

# Ejercicio 9:
for k in range(1, 7):
    top = buscar("seguridad", k=k)
    medio = sum(s for _, s, _ in top) / len(top)
    print(f"k={k} -> score medio {medio:.3f}")

# Ejercicio 10:
casos10 = casos + [{"pregunta": "¿Cuántos días de vacaciones?", "esperado": "23"}]
media = sum(precision_contexto(c["pregunta"], c["esperado"]) for c in casos10) / len(casos10)
print("Precisión de contexto media:", round(media, 2))

# Ejercicio 11:
def rag_langchain_fuentes(pregunta):
    docs = retriever.invoke(pregunta)
    contexto = "\n".join(d.page_content for d in docs)
    resp = llm.invoke(prompt.format(context=contexto, question=pregunta)).content
    return resp, [d.page_content for d in docs]

# Ejercicio 12:
SISTEMA_ESTRICTO = (
    "Responde solo con datos presentes en el contexto. Está prohibido inventar. "
    "Si el dato no aparece, responde exactamente 'No consta en la documentación'."
)
# Ante una pregunta fuera del corpus (p.ej. '¿Cuál es el color corporativo?'),
# la respuesta correcta sería 'No consta en la documentación'.


In [ ]:
# --- Soluciones a los DRILLS ---

# Drill 1:
m = SentenceTransformer("all-MiniLM-L6-v2")
print("Dimensión:", m.get_sentence_embedding_dimension())

# Drill 2:
v = model.encode(["energía solar"])[0]
vn = model.encode(["energía solar"], normalize_embeddings=True)[0]
print("Norma sin normalizar:", float(np.linalg.norm(v)),
      "| normalizada:", float(np.linalg.norm(vn)))

# Drill 3:
def coseno_drill(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

# Drill 4:
texto500 = "a" * 500
print("Chunks 100 sin solape:", len(trocear_fijo(texto500, 100)))

# Drill 5:
print("Chunks 100 solape 25:", len(trocear_con_solape(texto500, 100, 25)))

# Drill 6:
mx = model.encode(["uno", "dos", "tres", "cuatro"])
print("Forma:", mx.shape)

# Drill 7:
qd = model.encode(["horario"], normalize_embeddings=True)[0]
print("Chunk más similar:", int(np.argmax(matriz @ qd)))

# Drill 8:
idx_d = faiss.IndexFlatIP(matriz.shape[1])
idx_d.add(matriz)
print("ntotal:", idx_d.ntotal)

# Drill 9:
cli = chromadb.EphemeralClient()
col = cli.create_collection("demo")
col.add(ids=["a", "b", "c"], documents=["hola", "adiós", "buenos días"])

# Drill 10:
print(col.query(query_texts=["saludo"], n_results=1)["documents"])

# Drill 11:
def construir_prompt_min(pregunta, contexto):
    return f"Contexto: {contexto}\nPregunta: {pregunta}\nResponde solo con el contexto."

# Drill 12:
llamada = dict(
    model="llama3.2",
    messages=[
        {"role": "system", "content": "Responde solo con el contexto."},
        {"role": "user", "content": "¿Cuál es el horario?"},
    ],
)
# ollama.chat(**llamada)  # descomentar con Ollama en marcha
